[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/k_surfaces/k_surfaces.ipynb)

# Discrete Saddle Surfaces

Surfaces of constant negative curvature bend one way along one direction and the other way across it, like a saddle everywhere. Their discrete versions, K-surfaces, are grown from two crossing curves of surface normals: every missing normal follows from three known ones, and the whole patch follows from the normals.

Each normal is a plane, a vector, and a quadrilateral is completed by reflecting its opposite normal in the plane that bisects its two neighbours. The edge between two neighbouring points is the line where their tangent planes cross, `radius * normal.commutator(neighbour)`, an antivector like the points themselves, with an intrinsic radius setting its length and its sense. The same construction run in one more direction gives a second surface joined to the first by equal segments tangent to both, its Bäcklund partner.

The first animation bends the saddle while every grid edge keeps its length; the second circles a saddle and its partner; and two partners complete to one common fourth surface, either way round.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA3D as ga
from examples.util.animation import show_animation
from examples.surfaces.k_surfaces.core import normal_net, partner
from examples.surfaces.k_surfaces import render

In [ ]:
mv = NumpyContext(ga).multivector
# Normals are planes, vectors; points and the edges between them are antivectors.
Normal = ga.gatype.vector()
Point = ga.gatype.antivector()
Antipseudoscalar = ga.gatype.scalar().derive.flipped_orientation

## 1. Bending a saddle

Changing the angle between the two central curves of normals changes the shape while keeping every grid edge's length; the quadrilaterals change their angles and diagonals instead. Teal and copper mark the surface's two sides, and the camera stays fixed.

In [ ]:
n_half = 12
n_vertices = 2 * n_half + 1
extent = 0.8
curvature_radius = 1
neutral_angle = np.pi / 2
bend_amplitude = np.pi / 6
n_frames = 48
duration_ms = 70
phases = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
corner_angles = neutral_angle + bend_amplitude * np.sin(phases)
corner_extremes = neutral_angle + bend_amplitude * np.array([-1, 1])
angles = np.linspace(-extent, extent, n_vertices)
alternating = 1 - 2 * ((np.arange(n_vertices) - n_half) % 2)
reference_normal = mv.vector([0, 0, 1])                         # Normal
frame_volume = mv.x ^ mv.y ^ mv.z                                # Trivector
# The ordered frame gives the length scale its orientation.
radius = curvature_radius * frame_volume.dual()                  # Antipseudoscalar
origin = reference_normal.dual() * 0                             # Point

In [ ]:
def complete(base: Normal, left: Normal, right: Normal) -> Normal:
    # Reflect across the plane bisecting the two neighbouring unit normals.
    reflection = (left - right).normalized() >> Normal           # Normal <- Normal
    return reflection(base)


def surface(normals: Normal, radius: Antipseudoscalar, origin: Point) -> Point:
    # Each normal gives a map into its tangent plane, and its neighbour supplies the edge.
    edge_map = radius * Normal.commutator(normals)               # [rows, columns] Point <- Normal
    edges_u = edge_map[:-1, :](normals[1:, :])                   # [rows - 1, columns] Point
    edges_v = edge_map[:, :-1](normals[:, 1:])                   # [rows, columns - 1] Point
    # The edges close around each cell, so either route reaches the same point.
    row_origins = concatenate([origin[None], origin[None] + edges_u[:, 0].cumsum(axis=0)])
    offsets = concatenate([edges_v[:, :1] * 0, edges_v.cumsum(axis=1)], axis=1)
    return row_origins[:, None] + offsets


def saddle(corner_angle: float) -> tuple[Normal, Point]:
    second_plane = (mv.xy * (-corner_angle / 2)).exp() >> mv.yz  # Bivector
    mid_u = (mv.yz * (-angles / 2)).exp() >> reference_normal    # [vertices] Normal
    # Alternating signs let the same reflection rule serve both grid directions.
    mid_v = alternating * ((second_plane * (-angles / 2)).exp() >> reference_normal)   # [vertices] Normal
    normals = normal_net(mid_u, mid_v, complete)                 # [vertices, vertices] Normal
    points = surface(normals, radius, origin)                    # [vertices, vertices] Point
    return normals, points - points[n_half, n_half]


reference = stack([saddle(angle)[1] for angle in corner_extremes])   # [extremes, vertices, vertices] Point
states = (saddle(angle)[1] for angle in corner_angles)
show_animation(render.frames(states, reference), "k_surfaces_saddle", duration_ms)

## 2. A tangent partner

Choosing one new normal at a corner generates a whole second saddle: the copper sheet is the Bäcklund partner of the teal one. The dark segments join matching points; all have the same length, and each lies in the tangent planes at both its ends. The surfaces stay fixed while the camera moves around them, and the segments show faintly through the sheets so that their ends can be followed. This animation takes about ten seconds to draw.

In [ ]:
seed_azimuth = 1.6
seed_turn = 0.65
n_orbit_frames = 72
orbit_duration_ms = 100
orbit_elevation = 26
orbit_azimuth_centre = -70
orbit_azimuth_amplitude = 50
orbit_phases = np.linspace(0, 2 * np.pi, n_orbit_frames, endpoint=False)
orbit_azimuths = orbit_azimuth_centre + orbit_azimuth_amplitude * np.cos(orbit_phases)

In [ ]:
normals, points = saddle(neutral_angle)
corner_normal = normals[0, 0]                                    # Normal
reference_plane = (corner_normal ^ mv.x).normalized()            # Bivector
spin_plane = corner_normal | frame_volume                        # Bivector
seed_plane = (spin_plane * (-seed_azimuth / 2)).exp() >> reference_plane   # Bivector
starting_normal = (seed_plane * (-seed_turn / 2)).exp() >> corner_normal   # Normal

# The same cell-completion rule propagates from the chosen starting normal.
partner_normals = partner(normals, starting_normal, complete)    # [vertices, vertices] Normal
# The edge map also gives the step to the second sheet.
edge_map = radius * Normal.commutator(normals)                   # [vertices, vertices] Point <- Normal
partner_points = points + edge_map(partner_normals)              # [vertices, vertices] Point

show_animation(render.partner_orbit(partner_points, points, orbit_azimuths, orbit_elevation),
               "k_surfaces_backlund", orbit_duration_ms)

## 3. Composing transformations

Two partners of the same saddle determine a common fourth surface, and completing it from either partner agrees point by point: Bianchi permutability. The reflection that fills a grid cell is also what completes this pair of transformations.

In [ ]:
seed_turns = np.array([0.25, 0.7])

In [ ]:
starting_normals = (seed_plane * (-seed_turns / 2)).exp() >> corner_normal   # [partners] Normal
first_normals, second_normals = partner(normals, starting_normals, complete)
common_normals = complete(normals, first_normals, second_normals)   # [vertices, vertices] Normal
first_points = points + edge_map(first_normals)                  # [vertices, vertices] Point
second_points = points + edge_map(second_normals)                # [vertices, vertices] Point
via_first = first_points + radius * common_normals.commutator(first_normals)     # [vertices, vertices] Point
via_second = second_points + radius * common_normals.commutator(second_normals)  # [vertices, vertices] Point

In [ ]:
# checks
# The two completions meet; the partner's segments all have one length and lie in both tangent planes.
np.testing.assert_allclose((via_first - via_second).kernel, 0, atol=1e-10)
connectors = partner_points - points                             # [vertices, vertices] Point
np.testing.assert_allclose(connectors.norm().to_array(), connectors[0, 0].norm().to_array(), atol=1e-10)
np.testing.assert_allclose((normals & connectors).to_array(), 0, atol=1e-10)
np.testing.assert_allclose((partner_normals & connectors).to_array(), 0, atol=1e-10)

The construction follows Bobenko and Suris, [*Discrete Differential Geometry: Consistency as Integrability*, sections 2.4–2.5](https://brickisland.net/ddg-web/readings/pdf/DDGConsistencyAsIntegrability.pdf#page=58).